In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly.express as px


archivos = {
    '2016-17': 'LaLiga16 17.xlsx',
    '2017-18': 'LaLiga17 18.xlsx',
    '2018-19': 'LaLiga18 19.xlsx',
    '2019-20': 'LaLiga19 20.xlsx',
    '2020-21': 'LaLiga20 21.xlsx',
    '2021-22': 'LaLiga21 22.xlsx'
}

dfs = []

for temporada, archivo in archivos.items():
    df_temp = pd.read_excel(archivo)
    df_temp['Temporada'] = temporada
    dfs.append(df_temp)

In [24]:
df = pd.concat(dfs, ignore_index=True)

df[['Goles_Local', 'Goles_Visitante']] = df['Score'].str.split(r'[-–]', expand=True).astype(float)
df['DGlocal'] = df['Goles_Local'] - df['Goles_Visitante']

condiciones = [
    df['DGlocal'] > 0,
    df['DGlocal'] == 0,
    df['DGlocal'] < 0
]

resultados = ['Local', 'Empate', 'Visitante']
df['Resultado'] = np.select(condiciones, resultados, default=None)

puntos_local = [3, 1, 0]
df['Puntos_Local'] = np.select(condiciones, puntos_local, default=np.nan)

puntos_visita = [0, 1, 3]
df['Puntos_Visita'] = np.select(condiciones, puntos_visita, default=np.nan)

print(f'Dimensiones del dataset consolidado: {df.shape}')
display(df.head())

Dimensiones del dataset consolidado: (2527, 21)


,Wk,Day,Date,Time,Local,Score,Visitante,Attendance,Venue,Referee,...,Notes,Temporada,xG,xG.1,Goles_Local,Goles_Visitante,DGlocal,Resultado,Puntos_Local,Puntos_Visita
0,1.0,Vie,2016-08-19,20:45 (15:45),Málaga,1–1,Osasuna,22.347,Estadio La Rosaleda,Santiago Jaime,...,NaN,2016-17,NaN,NaN,1.0,1.0,0.0,Empate,1.0,1.0
1,1.0,Vie,2016-08-19,22:00 (17:00),La Coruña,2–1,Eibar,21.441,Estadio Municipal de Riazor,Mario Melero,...,NaN,2016-17,NaN,NaN,2.0,1.0,1.0,Local,3.0,0.0
2,1.0,Sáb,2016-08-20,18:15 (13:15),Barcelona,6–2,Betis,65.731,Camp Nou,Alberto Undiano,...,NaN,2016-17,NaN,NaN,6.0,2.0,4.0,Local,3.0,0.0
3,1.0,Sáb,2016-08-20,20:15 (15:15),Granada,1–1,Villarreal,15.149,Estadio Nuevo Los Cármenes,Javier Estrada,...,NaN,2016-17,NaN,NaN,1.0,1.0,0.0,Empate,1.0,1.0
4,1.0,Sáb,2016-08-20,22:15 (17:15),Sevilla,6–4,Espanyol,29.420,Estadio Ramón Sánchez Pizjuán,José González,...,NaN,2016-17,NaN,NaN,6.0,4.0,2.0,Local,3.0,0.0


# Pregunta 1
¿Cómo evoluciona la ventaja de la localía entre temporadas?

In [25]:
resumen_temporada = df.groupby('Temporada').agg(
    Partidos=('Resultado', 'count'),
    Victorias_Locales=('Resultado', lambda x: (x == 'Local').sum()),
    Empates=('Resultado', lambda x: (x == 'Empate').sum()),
    Victorias_Visitantes=('Resultado', lambda x: (x == 'Visitante').sum()),
    Goles_Local_Promedio=('Goles_Local', 'mean'),
    Goles_Visitante_Promedio=('Goles_Visitante', 'mean'),
    Puntos_Local_Promedio=('Puntos_Local', 'mean'),
    Puntos_Visitante_Promedio=('Puntos_Visita', 'mean'),
    DGlocal_Promedio=('DGlocal', 'mean')
)

resumen_temporada['%_Victorias_Locales'] = (resumen_temporada['Victorias_Locales'] / resumen_temporada['Partidos']) * 100
resumen_temporada['%_Empates'] = (resumen_temporada['Empates'] / resumen_temporada['Partidos']) * 100
resumen_temporada['%_Victorias_Visitantes'] = (resumen_temporada['Victorias_Visitantes'] / resumen_temporada['Partidos']) * 100

columnas_mostrar = [
    '%_Victorias_Locales', '%_Empates', '%_Victorias_Visitantes',
    'Goles_Local_Promedio', 'Goles_Visitante_Promedio',
    'Puntos_Local_Promedio', 'Puntos_Visitante_Promedio',
    'DGlocal_Promedio'
]

tabla_localia = resumen_temporada[columnas_mostrar].round(2)
display(tabla_localia)

,%_Victorias_Locales,%_Empates,%_Victorias_Visitantes,Goles_Local_Promedio,Goles_Visitante_Promedio,Puntos_Local_Promedio,Puntos_Visitante_Promedio,DGlocal_Promedio
Temporada,,,,,,,,
2016-17,47.63,23.42,28.95,1.66,1.28,1.66,1.10,0.38
2017-18,47.11,22.63,30.26,1.55,1.15,1.64,1.13,0.40
2018-19,44.21,28.95,26.84,1.45,1.13,1.62,1.09,0.32
2019-20,45.79,27.63,26.58,1.44,1.04,1.65,1.07,0.39
2020-21,41.58,28.68,29.74,1.37,1.14,1.53,1.18,0.23
2021-22,43.42,29.21,27.37,1.42,1.08,1.59,1.11,0.34


# 1. ¿En qué temporada la localía parece más fuerte?
La ventaja de jugar en casa fue más evidente durante la temporada 2016-17, donde se registró el mayor porcentaje de victorias locales (47.63%). Además, en la temporada 2017-18 se observó la mayor diferencia promedio de goles a favor del local (DGlocal = 0.40).

# 2. ¿En qué temporada parece más débil y qué impacto tuvieron las restricciones?
La localía fue notablemente más debil durante la temporada 2020-21. Los datos muestran una caída drástica en las victorias locales (41.58%) y la diferencia de goles local alcanzó su punto más bajo (0.23). Este cambio es muy relevante porque coincide exactamente con las restricciopnes de asistencia a los estadios debido a la pandemia, lo que sugiere que la ausencia de público disminuyó significativamente la ventaja de jugar en casa.

# Evidencia Descriptiva que Respalda la Conclusión
La conclusión sobre la disminución de la ventaja de localía debido a la ausencia de público se respalda concretamente en la siguiente evidencia descriptiva extraída de nuestros datos:

- Caída en la tasa de victorias: Los datos muestran una caída drástica en las victorias locales durante la temporada 2020-21, alcanzando un mínimo del 41.58% frente a promedios que históricamente rondaban el 45% al 47%.

- Reducción del margen de goles: La diferencia promedio de goles a favor del equipo local (DG_local_Promedio) alcanzó su punto más bajo (0.23) en esa misma temporada de pandemia, reduciéndo casi a la mitad en comparación con la temporada 2017-18 (0.40).

- Tendencia sostenida (Gráfico): Como se observa en la visualización del promedio móvil de 5 jornadas, el debilitamiento de la localía no fue un evento aislado de un par de fechas. La curva de la temporada de 2020-21 se ubica constantemente por debajo de las demás a lo largo de casi todo el campeonato.

In [38]:
fig = px.line(
    df_tendencia,
    x='Wk_Num',
    y='DGlocal_Promedio_Movil',
    color='Temporada',
    title='Evolución de la Ventaja de Localía a lo largo del Campeonato',
    labels={
        'Wk_Num': 'Jornada del Campeonato',
        'DGlocal_Promedio_Movil': 'Diferencia de Goles Local',
        'Temporada': 'Temporada'
    },
    template='plotly_white'
)

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.7)

fig.update_layout(
    xaxis=dict(range=[1, 38]),
    legend=dict(title='Temporada', orientation='v', yanchor='top', y=1, xanchor='left', x=1.02),
    hovermode='x unified'
)
fig.show()